# Немного о многоруких бандитах

В ноутбуке про многоруких бандитов мы разбирали классы для реализации отдельных стратегий решения задачки. Реализуйте в той же парадигме, что и в ноутбуке про бандитов, стратегии [exp3](https://jamesrledoux.com/algorithms/bandit-algorithms-epsilon-ucb-exp-python/),[softmax](https://gibberblot.github.io/rl-notes/siЁngle-agent/multi-armed-bandits.html) и **successive elimination** (прочитайте о ней в книжке [Сливкинса](https://arxiv.org/abs/1904.07272)). Надо понять, как оно работает и сделать так, чтобы оно работало в использованной нами конфигурации)

В конце концов, вы ж на работе тоже нон-стоп гуглите)

# Еще про бандитов

У [Сливкинса](https://arxiv.org/abs/1904.07272) разберитесь с вопросом нижней границы Regret'a для любой стратегии. Тут прикладывать ничего не нужно, просто возьмите и почитайте.

# ANN

Мы разобрали LSH, IVFPQ, Annoy и HNSW

Ниже (после кода) описание задачки 


In [3]:
# !pip install datasets

In [7]:
import time, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

SEED = 7
rng = np.random.default_rng(SEED)

# 1) Датасет
ds_train = load_dataset("ag_news", split="train")   # ~120k
ds_test  = load_dataset("ag_news", split="test")    # ~7.6k

X_train_text = [r["text"].strip() for r in ds_train]
y_train      = np.array([int(r["label"]) for r in ds_train], dtype=np.int64)
X_test_text  = [r["text"].strip() for r in ds_test]
y_test       = np.array([int(r["label"]) for r in ds_test], dtype=np.int64)

# 2) Эмбеддинги 
MODEL_NAME = "bert-base-nli-mean-tokens"
encoder = SentenceTransformer(MODEL_NAME)



X_db = encoder.encode(X_train_text, show_progress_bar = True)  # база (DB)
X_q  = encoder.encode(X_test_text, show_progress_bar = True)   # запросы (Q)
d = X_db.shape[1]

print("DB:", X_db.shape, "Q:", X_q.shape, "dim:", d)

# 3) (опц.) делаем дисбаланс в БАЗЕ для класса-«хвоста»
RARE_CLASS = 3
keep_ratio = 0.15  # оставим ~15% редкого класса в базе
idx_rare = np.where(y_train == RARE_CLASS)[0]
idx_keep_rare = rng.choice(idx_rare, size=max(1,int(len(idx_rare)*keep_ratio)), replace=False)
idx_nonrare = np.where(y_train != RARE_CLASS)[0]
idx_final = np.sort(np.concatenate([idx_nonrare, idx_keep_rare]))
X_db = X_db[idx_final]; y_db = y_train[idx_final]
print(f"DB after imbalance: {len(X_db)}; rare count={np.sum(y_db==RARE_CLASS)}")

# 4) Эталонный точный индекс (FlatIP) и топ-k для всех запросов
K = 50
flat = faiss.IndexFlatIP(d)
flat.add(X_db)

Batches:   0%|          | 0/3750 [00:00<?, ?it/s]

Batches:   0%|          | 0/238 [00:00<?, ?it/s]

DB: (120000, 768) Q: (7600, 768) dim: 768
DB after imbalance: 94500; rare count=4500


## Про "потерю информации" от ANN

Для HNSW, IVFPQ и Annoy (LSH не обязательно, но можно) оценить:
 

### 1) Структурные потери ANN

Оцениваем, насколько набор соседей из ANN совпадает с обычным kNN:
$$
\operatorname{Recall@k}^{\text{w.r.t. exact}} ;=; \frac{|A_k \cap E_k|}{k},
$$

где $(E_k)$ — точные top-(k) соседи, $(A_k)$ — top-(k) из ANN.
Чем ниже значение, тем больше потеря структуры соседства.

### 2) Потери информации о метках (классах)

Предсказываем метку по **голосованию** среди соседей и сравниваем с эталоном:
$$
\Delta \mathrm{Acc@k} = \mathrm{Acc}\big(\hat y^{\text{ANN}}*{k}\big)-\mathrm{Acc}\big(\hat y^{\text{Exact}}*{k}\big),
$$
где $(\hat y_k)$ — метка, выбранная по большинству среди top-(k) соседей.


### 3) Неоднородность распределения классов среди соседей

Для каждого запроса считаем частоты классов среди соседей: $(p^{\text{Exact}}*{k})$ и $(p^{\text{ANN}}*{k})$.

В качестве меры неоднородности распределения классов можете использовать, что вам заблагорассудится: можете энтропию, можете Джини, а можете еще что-то :)


### 4) Потери по «хвосту» (редкий класс)

Выбираем «редкий» класс и искусственно уменьшаем его долю в базе.
Считаем Rare-Recall@k как долю запросов редкого класса, у которых **есть хотя бы один** сосед того же класса в top-(k).
Затем смотрим разницу относительно эталона – Flat-индекса:
$$
\Delta \mathrm{Rare\text{-}Recall@k} = \mathrm{Rare\text{-}Recall@k}^{\text{ANN}} - \mathrm{Rare\text{-}Recall@k}^{\text{Exact}}.
$$

### 5) Потеря  близости

Сравниваем среднюю косинусную близость (или скалярное произведение) соседей у эталона и ANN:

$$
\Delta \overline{s} = \overline{\langle q, x_{kNN}\rangle}-\overline{\langle q, x_{ANN}\rangle}.
$$


Положительное $(\Delta \overline{s})$ означает падение средней близости у ANN относительно эталона.

---

### Какой ANN лучше?

Какой из ANN-индексов на ваш взгляд получился лучше?
